# IaC 第1周:Terraform 基础 — 用代码管理基础设施

> **学习目标**:理解声明式 IaC 的核心价值,掌握 Terraform 工作流(init/plan/apply/destroy),能用 HCL 语法管理 Docker 资源

---

## Day 1:为什么需要 IaC

### ClickOps vs GitOps

| 维度 | ClickOps (手动) | GitOps (IaC) |
|------|----------------|--------------|
| 可复现 | 依赖记忆和截图 | `terraform apply` 即可 |
| 版本管理 | 无("上次怎么配的？") | Git 提交历史完整可追溯 |
| Code Review | 无 | PR review 把关变更 |
| 变更回滚 | 手动改回去 | `git revert` + `terraform apply` |
| 审计 | 需要查操作日志 | Git log 就是审计日志 |

### 声明式 vs 命令式

```
命令式 (Ansible, Shell 脚本):
  1. 创建 VPC
  2. 创建子网
  3. 创建安全组
  4. 创建 EC2
  → 你要告诉它每一步怎么做

声明式 (Terraform, K8s YAML):
  我想要:
    - 1 个 VPC
    - 2 个子网
    - 1 个安全组 (允许 443)
    - 3 台 EC2 (t3.medium)
  → 你只描述想要什么,工具自己算出怎么做
```

## Day 2:Terraform 核心工作流

| 步骤 | 命令 | 说明 |
|------|------|------|
| 1. 编写配置 | `main.tf` | 声明期望的资源状态 |
| 2. 初始化 | `terraform init` | 下载 provider 插件、初始化 backend |
| 3. 计划 | `terraform plan` | 对比当前状态和期望状态,生成执行计划 |
| 4. 应用 | `terraform apply` | 执行计划,创建/修改/删除资源 |
| 5. 销毁 | `terraform destroy` | 删除所有 Terraform 管理的资源 |

## Day 3:HCL 语法基础

```hcl
# variables.tf
variable "environment" {
  type    = string
  default = "dev"
  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment must be dev, staging, or prod"
  }
}

# main.tf
terraform {
  required_providers {
    docker = {
      source  = "kreuzwerker/docker"
      version = "~> 3.0"
    }
  }
}

provider "docker" {}

resource "docker_container" "web" {
  name  = "web-${var.environment}"
  image = "nginx:latest"
  ports {
    internal = 80
    external = var.external_port
  }
}

# outputs.tf
output "access_url" {
  value = "http://localhost:${var.external_port}"
}
```

核心语法元素:`resource`(资源)、`variable`(变量)、`output`(输出)、`data`(数据源)、`locals`(局部值)

## Day 4:State 文件

`terraform.tfstate` 是 Terraform 的核心:

1. **映射**:记录 .tf 文件中的 resource 和真实资源的对应关系
2. **缓存**:plan 时对比 state 和期望配置,计算差异
3. **元数据**:记录资源依赖和 provider 信息

State 管理最佳实践:
- 永远不要手动编辑 state 文件
- 使用 Remote Backend (S3/GCS) 存储 state —— 多人协作必须
- 启用 State Locking(S3 用 DynamoDB)
- state 文件包含敏感信息(密码、密钥),不要提交到 Git

## Day 5:依赖图

Terraform 通过分析资源引用关系自动构建 DAG(有向无环图),决定创建/删除顺序:

```
docker_image.nginx (无依赖)       docker_network.app (无依赖)
        │                                    │
        └──────────┬─────────────────────────┘
                   │
        docker_container.nginx (依赖: image + network)
```

创建顺序:无依赖的先创建。销毁顺序:正好相反。Terraform 自动推导,无需手动指定

## Day 6:变量赋值方式与优先级

| 优先级 | 方式 | 示例 |
|--------|------|------|
| 1 (最高) | CLI 参数 | `terraform apply -var='env=prod'` |
| 2 | *.auto.tfvars | 自动加载 |
| 3 | terraform.tfvars | 默认自动加载 |
| 4 | 环境变量 | `export TF_VAR_env=prod` |
| 5 (最低) | default | variable 块中的默认值 |

最佳实践:secret 变量永远不要写默认值或 tfvars,用环境变量或 Vault

## Day 7:第1周综合练习

In [ ]:
print("=" * 60)
print("第1周综合练习:Terraform 管理 Docker 应用栈")
print("=" * 60)

print("""
项目结构:
terraform-docker/
├── main.tf
├── variables.tf
├── outputs.tf
└── terraform.tfvars

应该管理的资源:
  ├── docker_network.app_net        # 自定义网络
  ├── docker_volume.pg_data          # PostgreSQL 持久化
  ├── docker_container.nginx         # nginx 容器 (端口 8080)
  ├── docker_container.web           # web 容器 (端口 5000)
  └── docker_container.postgres      # PostgreSQL 容器 (端口 5432)

要求:
  - 所有端口可配置 (变量)
  - postgres 密码为 sensitive 变量
  - output 导出 nginx 访问 URL

验证命令:
  terraform fmt -recursive     # 格式化
  terraform validate           # 语法检查
  terraform plan               # 查看执行计划
  terraform apply              # 创建所有资源
  terraform destroy            # 清理所有资源
""")

print("=" * 60)
print("第1周核心收获:")
print("1. IaC 的核心价值:可复现、可版本管理、可 Code Review")
print("2. terraform init → plan → apply → destroy 工作流")
print("3. HCL 语法: resource / variable / output / data / locals")
print("4. State 是 Terraform 的核心:映射配置到真实资源")
print("5. Terraform 自动分析资源引用,生成 DAG 决定创建/删除顺序")
print("=" * 60)